# Projeto: Orquestração de Great Expectation Camada Silver - Chuvas Criticas

---

### Objetivo
Este notebook centraliza a execução das etapas de validação automatizada dos dados climáticos da Chuvas Criticas e persistidos no na camada *silver*, utilizando a biblioteca `Great Expectations` e um esquema padronizado de verificação de qualidade. O objetivo principal é garantir a confiabilidade e consistência das tabelas antes de seu avanço nas próximas camadas da arquitetura.

## Painel de Alerta
Demonstra um painel com o percentual de expectativas bem-sucedidas com base nas estatísticas retornadas pela execução da validação.

`classificar_status(percentual)`
Classifica visualmente o status da tabela conforme o percentual de sucesso:
- **🟢 100%** Success
- **🟡 ≥ 60%** Fail
- **🔴 < 60%** Fail

---

## Histórico de Atualizações

| Data       | Responsável    | Versão | Descrição                                                                 |
|------------|----------------|--------|---------------------------------------------------------------------------|
| 16/05/2025 | Evelise Dias    | 1.0    | Estruturação Inicial do projeto utilizando assets e suites     |
| 20/05/2025 | Evelise Dias   | 1.2    | Refatoração do projeto utilizando abordagem diferente para cumprir com os requisitos.    |
| 20/05/2025 | Evelise Dias   | 1.3   | Inclusão de Documentação, docstrings, sumário e organização do notebook.    |
| 21/05/2025 | Evelise Dias   | 1.4   | Inclusão de função que interrompe a execução caso classificação seja menor que 80%.    |
| 24/05/2025 | Evelise Dias   | 1.5   | Refatoração do código gerando df_validados e persistir na camada gold e df_quarentena para identificar dados descartados.|
| 25/05/2025 | Evelise dias   | 1.6   | Refatoração do código gerando funções auxiliares e centralizando nb para execução |

In [0]:
%run ./funcoes/funcoes_gx 

In [0]:
"""
Executa a validação da tabela 'chuvas_criticas' na camada silver, utilizando Great Expectations,
e separa os dados validados e não validados (quarentena), salvando-os em formato Delta 
em seus respectivos diretórios.

Processo:
1. Chama a função `validar_tabelas_silver` com os parâmetros do catálogo, camada, tabela
   e caminho para o contexto do Great Expectations.
2. Recupera os DataFrames resultantes da validação:
   - `df_validado`: registros que passaram em todas as regras.
   - `df_quarentena`: registros que falharam em alguma validação.
3. Salva os dados validados e de quarentena separadamente em formato Delta Lake.

Parâmetros:
- catalogo (str): Nome do catálogo do Unity Catalog.
- camada (str): Nome da camada (ex: 'silver').
- tabelas (list): Lista com o nome das tabelas a serem validadas.
- contexto_path (str): Caminho para o contexto do Great Expectations.

Nota:
- Essa abordagem evita a conversão para Pandas, preservando a computação distribuída com Spark.
- Warnings gerados durante o processo são relacionados a bibliotecas de terceiros e não afetam o resultado final da execução.
"""
pasta_logs = lakehouse_logs_gx + "/silver/chuvas_criticas"

resultado_validacao = validar_tabelas_silver(
    catalogo="grupo01",
    camada="silver",
    tabelas=["chuvas_criticas"],
    contexto_path=caminho_ge,
    pasta_logs = pasta_logs
)

df_chuvas_criticas_validado = resultado_validacao["grupo01.silver.chuvas_criticas"]["df_validado"]
df_chuvas_criticas_quarentena = resultado_validacao["grupo01.silver.chuvas_criticas"]["df_quarentena"]

pasta_quarentena = lakehouse_logs_gx + "/silver/chuvas_criticas/quarentena/chuvas_criticas"
pasta_validos = lakehouse_logs_gx + "/silver/chuvas_criticas/validos/chuvas_criticas"

df_chuvas_criticas_validado = df_chuvas_criticas_validado.dropDuplicates(["nom_municipio", "sig_uf", "vol_max_chuva", "pais"])
df_chuvas_criticas_quarentena = df_chuvas_criticas_quarentena.dropDuplicates(["nom_municipio", "sig_uf", "vol_max_chuva", "pais"])

df_chuvas_criticas_validado.write.mode("overwrite").format("delta").save(pasta_validos)
df_chuvas_criticas_quarentena.write.mode("overwrite").format("delta").save(pasta_quarentena)
# escreve_com_merge_delta(
#     dataframe=df_chuvas_criticas_validado,
#     caminho_tabela_delta=pasta_validos,
#     campos_chave=["dt_carga", "nom_municipio", "sig_uf", "vol_max_chuva", "pais"]
# )
# escreve_com_merge_delta(
#     dataframe=df_chuvas_criticas_quarentena,
#     caminho_tabela_delta=pasta_quarentena,
#     campos_chave=["dt_carga", "nom_municipio", "sig_uf", "vol_max_chuva", "pais"]
# )

In [0]:
exibir_resumo_validacoes(pasta_logs)

In [0]:
mensagem = montar_mensagem(pasta_logs)
webhook_url = "https://cwisw.webhook.office.com/webhookb2/02f1a3c3-69cf-40ad-9fdb-d1b024f3d9cd@647631af-8bf8-4048-a98f-b1fbee134a6d/IncomingWebhook/7955a3c1951d4916856f1fadc1e4a547/d5dc3b64-26d6-495d-ab66-85ff66166d12/V2aRlSVPfgXum0iawCcpgcSXg4ZbNuGEpQBYY1mgFa9js1"
enviar_alerta_teams(mensagem, webhook_url)

In [0]:
df = spark.read.format("delta").load(lakehouse_logs_gx + "/silver/chuvas_criticas/validos/chuvas_criticas")
display(df)


In [0]:
display(df_chuvas_criticas_validado)